# Derive Andvari Summary

This notebook derives an aggregate summary from `data/exported/andvari_invocations.csv`.

It produces `data/derived/andvari_summary.csv` with one row per agent/variant pair, an agent-only total row where `Variant` is `All`, and a final grand total row.

In [ ]:
from pathlib import Path

import pandas as pd


def find_package_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'data' / 'exported' / 'andvari_invocations.csv').exists():
            return path
    raise FileNotFoundError('Could not find data/exported/andvari_invocations.csv')


PACKAGE_ROOT = find_package_root(Path.cwd())
INPUT_CSV = PACKAGE_ROOT / 'data' / 'exported' / 'andvari_invocations.csv'
OUTPUT_CSV = PACKAGE_ROOT / 'data' / 'derived' / 'andvari_summary.csv'

df = pd.read_csv(INPUT_CSV)
df.head()

## Validate Required Columns

The table is derived only from fields exported by the Andvari invocation table.

In [ ]:
required = [
    'agent',
    'variant',
    'status',
    'duration_seconds',
]

missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df[required].head()

## Build Summary Table

`Passed` counts invocations with `status == 'passed'`; `Errors` counts all other statuses. Runtime is reported as the median duration in minutes. Rates and runtimes are rounded to three decimals.

In [ ]:
work = df[required].copy()

work['duration_seconds'] = pd.to_numeric(work['duration_seconds'], errors='coerce')

work['passed_flag'] = work['status'].eq('passed')
work['error_flag'] = ~work['passed_flag']


def summarize(grouped: pd.core.groupby.DataFrameGroupBy) -> pd.DataFrame:
    return grouped.agg(
        Invocations=('status', 'size'),
        Passed=('passed_flag', 'sum'),
        Errors=('error_flag', 'sum'),
        median_runtime_seconds=('duration_seconds', 'median'),
    ).reset_index()


by_variant = summarize(work.groupby(['agent', 'variant'], dropna=False))
by_agent = summarize(work.groupby(['agent'], dropna=False))
by_agent.insert(1, 'variant', 'All')

total = summarize(work.groupby(lambda _: 'Total'))
total = total.drop(columns=['index'])
total.insert(0, 'agent', 'Total')
total.insert(1, 'variant', 'All')

summary = pd.concat([by_variant, by_agent, total], ignore_index=True)
summary = summary.rename(columns={'agent': 'Agent', 'variant': 'Variant'})
summary['Pass rate'] = (summary['Passed'] / summary['Invocations']).round(3)
summary['Median runtime (min)'] = (summary['median_runtime_seconds'] / 60).round(3)

summary[['Invocations', 'Passed', 'Errors']] = summary[
    ['Invocations', 'Passed', 'Errors']
].astype(int)

summary = summary[
    ['Agent', 'Variant', 'Invocations', 'Passed', 'Errors', 'Pass rate', 'Median runtime (min)']
]
summary['_sort_total'] = summary['Agent'].eq('Total')
summary = summary.sort_values(['_sort_total', 'Agent', 'Variant'], kind='stable')
summary = summary.drop(columns=['_sort_total']).reset_index(drop=True)
summary

## Export Derived CSV

The write step overwrites `data/derived/andvari_summary.csv` if it already exists.

In [ ]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_CSV, index=False)
OUTPUT_CSV